# KoFinance로 삼성전자 재무제표 분석하기

이 노트북에서는 KoFinance Python SDK를 사용하여 삼성전자(005930)의 재무제표를 조회하고,
매출/영업이익 추이와 수익성 지표를 시각화합니다.

In [ ]:
# 설치 (처음 한 번만)
# !pip install kofinance matplotlib

from kofinance import KoFinance
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (macOS)
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# KoFinance 클라이언트 초기화
import os

kf = KoFinance(os.environ.get("KOFINANCE_API_KEY", "your-api-key"))

## 1. 재무제표 조회

삼성전자의 최근 5년 연간 재무제표를 가져옵니다.

In [ ]:
# 삼성전자 재무제표 (최근 5년, 연간)
df = kf.financials("005930", period="5y", type="annual")

# 주요 컬럼 확인
df[["period", "is_revenue", "is_operating_income", "is_net_income",
    "ratio_roe", "ratio_per", "ratio_debt_ratio"]]

## 2. 매출액 및 영업이익 추이 시각화

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# 매출액 (막대 그래프)
bars = ax1.bar(
    df["period"],
    df["is_revenue"] / 1e12,
    alpha=0.7,
    color="#4A90D9",
    label="매출액 (조원)",
)

# 영업이익 (선 그래프, 보조 y축)
ax2 = ax1.twinx()
ax2.plot(
    df["period"],
    df["is_operating_income"] / 1e12,
    "r-o",
    linewidth=2,
    markersize=8,
    label="영업이익 (조원)",
)

ax1.set_xlabel("회계연도")
ax1.set_ylabel("매출액 (조원)", color="#4A90D9")
ax2.set_ylabel("영업이익 (조원)", color="red")

# 범례
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("삼성전자 매출액 및 영업이익 추이")
plt.tight_layout()
plt.show()

## 3. 수익성 지표 분석

In [ ]:
# 최근 연도 주요 지표
latest = df.iloc[0]
print("=" * 40)
print(f"  삼성전자 ({latest['period']}) 주요 지표")
print("=" * 40)
print(f"  ROE:     {latest['ratio_roe']:.1f}%")
print(f"  PER:     {latest['ratio_per']:.1f}배")
print(f"  부채비율: {latest['ratio_debt_ratio']:.1f}%")
print(f"  EPS:     {latest['ratio_eps']:,.0f}원")
print(f"  영업이익률: {latest['ratio_operating_margin']:.1f}%")
print("=" * 40)

In [ ]:
# ROE 추이 그래프
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROE 추이
axes[0].plot(df["period"], df["ratio_roe"], "g-o", linewidth=2, markersize=8)
axes[0].fill_between(df["period"], df["ratio_roe"], alpha=0.1, color="green")
axes[0].set_title("ROE 추이")
axes[0].set_ylabel("ROE (%)")
axes[0].set_xlabel("회계연도")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=10, color="gray", linestyle="--", alpha=0.5, label="10% 기준선")
axes[0].legend()

# 부채비율 추이
axes[1].bar(df["period"], df["ratio_debt_ratio"], color="#FF6B6B", alpha=0.7)
axes[1].set_title("부채비율 추이")
axes[1].set_ylabel("부채비율 (%)")
axes[1].set_xlabel("회계연도")
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=100, color="red", linestyle="--", alpha=0.5, label="100% 기준선")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 최근 공시 확인

In [ ]:
# 최근 30일 공시 조회
disclosures = kf.disclosures("005930", days=30)

# 주요 컬럼만 표시
disclosures[["date", "title", "type", "summary"]].head(10)

## 5. 동종업계 비교

삼성전자와 SK하이닉스의 재무제표를 비교합니다.

In [ ]:
# 삼성전자 + SK하이닉스 동시 조회
compare = kf.financials(["005930", "000660"], period="3y")

# ROE 비교
fig, ax = plt.subplots(figsize=(10, 5))

for symbol, group in compare.groupby("symbol"):
    name = group.iloc[0]["name"]
    ax.plot(group["period"], group["ratio_roe"], "-o", label=name, linewidth=2)

ax.set_title("삼성전자 vs SK하이닉스 ROE 비교")
ax.set_ylabel("ROE (%)")
ax.set_xlabel("회계연도")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 결론

KoFinance SDK를 사용하면:

- **재무제표 조회**: `kf.financials()` 한 줄로 손익계산서, 재무상태표, 현금흐름표, 주요 비율을 DataFrame으로 받을 수 있습니다.
- **공시 + AI 요약**: `kf.disclosures()`로 DART 공시와 AI 요약을 동시에 확인할 수 있습니다.
- **다중 종목 비교**: 종목코드 리스트를 넘기면 여러 기업을 한번에 비교할 수 있습니다.
- **시각화 연동**: pandas DataFrame을 반환하므로 matplotlib, plotly 등과 바로 연동됩니다.

API 키 발급: [kofinance.ntriq.co.kr](https://kofinance.ntriq.co.kr)  
전체 문서: [kofinance.ntriq.co.kr/docs](https://kofinance.ntriq.co.kr/docs)